# BinWise AI — RealWaste training
Use a free Colab GPU runtime. The notebook downloads the CC BY 4.0 RealWaste dataset, creates a seeded stratified split, trains MobileNetV3-Small, and optionally uploads measured artifacts to Hugging Face Hub.

In [ ]:
# Change this once after publishing the repository.
GITHUB_REPO = "https://github.com/YOUR_GITHUB_NAME/binwise-ai.git"
!git clone $GITHUB_REPO /content/binwise-ai
%cd /content/binwise-ai
!pip -q install -r requirements-train.txt

In [ ]:
import os, sys, torch
sys.path.insert(0, "/content/binwise-ai/src")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available — select a GPU runtime")

In [ ]:
!python scripts/download_realwaste.py --output data/raw
!PYTHONPATH=src python scripts/create_split.py data/raw/realwaste --output data/processed/split.csv --seed 42

In [ ]:
# Saved last_checkpoint.pt makes interrupted runs inspectable. Colab GPU memory varies; lower batch size if needed.
!PYTHONPATH=src python scripts/train.py --manifest data/processed/split.csv --output artifacts/model --batch-size 64 --workers 2

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display
metrics = json.loads(Path("artifacts/model/metrics.json").read_text())
print(json.dumps(metrics["test"], indent=2))
display(Image("artifacts/model/confusion_matrix.png"))

In [ ]:
# Optional: upload only after reviewing the measured metrics above.
from huggingface_hub import notebook_login
notebook_login()
HF_MODEL_REPO = "YOUR_HF_NAME/binwise-realwaste-mobilenet"
!PYTHONPATH=src python scripts/upload_model.py $HF_MODEL_REPO --folder artifacts/model